# Stage 2 Notebook 34 - Exp2CC Exp2Z + 30 epochs (convergence test)

**Why this exists.** Exp2Z (NB31, uncertainty weighting) showed monotonic improvement through 15 epochs:

| ep | val_lane_best_f1 | matched_iou | decoded_f1 | oracle_f1 |
|---|---:|---:|---:|---:|
| 1 | 0.659 | 0.114 | 0.041 | 0.064 |
| 5 | 0.661 | 0.139 | 0.027 | 0.068 |
| 10 | 0.672 | 0.155 | 0.044 | 0.096 |
| 15 | 0.681 | **0.162** | **0.044** | **0.101** |

Geometry (matched_iou) was *still climbing* at epoch 15, suggesting we may not have reached convergence. CLRKDNet trains 70+ epochs on CULane; we've been at 10-15.

Exp2CC = Exp2Z exactly + `end_epoch: 15 -> 30`. Tests whether:
- More epochs let geometry continue climbing toward Exp2N's 0.42 territory
- decoded_f1 keeps growing in lockstep with oracle_f1
- Or the trajectory plateaus (in which case the bottleneck is architectural, not training-budget)

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 30-epoch short run (~30 minutes total).
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp29_rmt_gca_mask_uncertainty_long30_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp29_rmt_gca_mask_uncertainty_long30_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp29_rmt_gca_mask_uncertainty_long30_joint_smoke.log
OK exp29_rmt_gca_mask_uncertainty_long30_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.0606 det_loss=3.4556 grad_cos=-0.1023 lambda_lane=0.0812
  gate_stats={'gate/det_mean': 0.5006389021873474, 'gate/lane_mean': 0.5038171410560608, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp29_rmt_gca_mask_uncertainty_long30_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'long30'
    EPOCHS = 30
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

流式输出内容被截断，只能显示最后 5000 行内容。
  "val/lane/decoded_oracle_pred_count": 48.0,
  "val/lane/decoded_oracle_avg_score": 0.14921983294188976,
  "val/lane/decoded_oracle_top_k_used": 6.0,
  "val/det/metric_map50": 0.003790612403827254,
  "val/det/metric_precision50": 0.015375000028870999,
  "val/det/metric_recall50": 0.14596827905625104,
  "val/det/metric_num_gt": 85.65,
  "val/det/metric_num_pred": 800.0,
  "val/lane/cls": 0.0685211039148271,
  "val/lane/cls_pos": 0.08083417192101479,
  "val/lane/cls_neg": 0.0562080361880362,
  "val/lane/reg": 0.17309833727777005,
  "val/lane/xytl": 0.1166448175907135,
  "val/lane/line_iou": 0.8570801734924316,
  "val/lane/mask_aux": 0.46092833057045934,
  "val/lane/smooth": 0.001462938418262638,
  "val/lane/distill": 0.0,
  "val/lane/geometry_raw": 1.148286271095276,
  "val/lane/geometry_weighted": 2.0322033882141115,
  "val/lane/unweighted_total": 1.6777357190847397,
  "val/lane/weighted_total": 3.228144496679306,
  "val/lane/total": 3.228144496679306,
  "val

0

## What to watch in Exp2CC training

Pass criteria at epoch 30:
- **`val/lane/decoded_oracle_f1 >= 0.15`**: oracle ceiling extends with longer training. Convergence-limited.
- **`val/matched_line_iou >= 0.20`**: geometry climbs past Exp2Z's 0.162.
- **`val/lane/decoded_f1 >= 0.06`**: cls/decode track geometry's improvement.

Failure signals:
- All metrics plateau by epoch 20 (no continued improvement): bottleneck is architectural, not duration. Move to Exp2DD (KD).
- Geometry continues improving but decoded_f1 doesn't: cls is genuinely the structural bottleneck; KD is the next move.